In [21]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import pickle
import joblib

# Load dataset
try:
    df = pd.read_csv("seattle-weather.csv")
except FileNotFoundError:
    print("Error: 'seattle-weather.csv' not found. Please ensure the file is in the same directory.")
    exit(1)

# Check for missing values
if df.isnull().sum().any():
    print("Warning: Missing values detected. Filling with mean values.")
    df.fillna(df.mean(numeric_only=True), inplace=True)

# Encode weather labels
le = LabelEncoder()
df['weather_encoded'] = le.fit_transform(df['weather'])
joblib.dump(le, "label_encoder.pkl")  # Save encoder

# Features & target (multi-class)
X = df[['precipitation', 'temp_max', 'temp_min', 'wind']]
y = df['weather_encoded']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Save model
with open("weather_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save predictions to CSV
df['predicted_weather'] = model.predict(X)
df['predicted_label'] = le.inverse_transform(df['predicted_weather'])
df.to_csv("weather_with_predict.csv", index=False)

print("Model and encoder saved. Predictions file generated.")

Classification Report:
              precision    recall  f1-score   support

     drizzle       0.20      0.11      0.14         9
         fog       0.40      0.08      0.13        25
        rain       0.92      0.92      0.92       120
        snow       0.40      0.25      0.31         8
         sun       0.78      0.95      0.86       131

    accuracy                           0.82       293
   macro avg       0.54      0.46      0.47       293
weighted avg       0.78      0.82      0.78       293

Confusion Matrix:
[[  1   0   0   0   8]
 [  1   2   2   0  20]
 [  0   0 110   3   7]
 [  0   0   6   2   0]
 [  3   3   1   0 124]]
Model and encoder saved. Predictions file generated.
